In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/neurips-open-polymer-prediction-2025/sample_submission.csv
/kaggle/input/neurips-open-polymer-prediction-2025/train.csv
/kaggle/input/neurips-open-polymer-prediction-2025/test.csv
/kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset2.csv
/kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset4.csv
/kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset1.csv
/kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset3.csv


In [2]:
#import the rest
from copy import deepcopy
#import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import mutual_info_regression

#import scipy
from scipy.stats import spearmanr
import scipy.stats as st

#import torch-molecule
#from torch_molecule.utils.search import ParameterType, ParameterSpec
#LSTM model
#from torch_molecule import LSTMMolecularPredictor
#print(f'torch-molecule imported')


#import torch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.data import Data, DataLoader, Batch
from torch_geometric.nn import MessagePassing, global_add_pool, global_mean_pool
print(f'torch imported')


#import rdkit
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import rdMolDescriptors  
print(f'rdkit imported')

#import tqdm to show progress
from tqdm import tqdm
print(f'tqdm imported')

ModuleNotFoundError: No module named 'torch_geometric'

In [ ]:
csv_path = '/kaggle/input/neurips-open-polymer-prediction-2025/train.csv'
train_df = pd.read_csv(csv_path)
print(train_df.shape)
print(train_df.head(2))

In [4]:
#train\test split

#20% ---> dev_test
temp_df, dev_test = train_test_split(
    train_df,
    test_size = 0.2,
    random_state = 42, #reproducbility
    shuffle = True
)

#80% ---> 75%train 25%val 
dev_train, dev_val = train_test_split(
    temp_df,
    test_size = 0.25,
    random_state = 42,
    shuffle = True
)

In [5]:
#check
print(f"Total rows:   {len(train_df)}")
print(f"Dev train:    {len(dev_train)} ({len(dev_train)/len(train_df):.2%})")
print(f"Dev valid:    {len(dev_val)} ({len(dev_val)/len(train_df):.2%})")
print(f"Dev test:     {len(dev_test)} ({len(dev_test)/len(train_df):.2%})")

Total rows:   7973
Dev train:    4783 (59.99%)
Dev valid:    1595 (20.01%)
Dev test:     1595 (20.01%)


In [6]:
print(dev_train.columns.tolist())

['id', 'SMILES', 'Tg', 'FFV', 'Tc', 'Density', 'Rg']


In [7]:
# check some sample
print(f"Polymer example: \n{dev_train['SMILES'].to_list()[:3]}\n")
print(f"Target columns: \n{dev_train.columns[2:].tolist()}\n")

Polymer example: 
['*Nc1ccc(CC(CC(C)(C)c2ccc(N*)cc2)=C(C)C)cc1', '*CC(*)(CC(=O)OC)C(=O)OC12CC3CC(C)(CC(C)(C3)C1)C2', '*OP(=O)(Oc1c(Cl)cc(Cl)cc1Cl)Oc1c(Cl)c(Cl)c(*)c(Cl)c1Cl']

Target columns: 
['Tg', 'FFV', 'Tc', 'Density', 'Rg']



using self-supervised GNN

In [8]:
# 1. 增强的图数据构建 ====================================================
def get_atom_features(atom):
    """原子级别特征（论文Table 1）"""
    features = [
        atom.GetAtomicNum(),
        int(atom.GetIsAromatic()),
        atom.GetDegree(),
        atom.GetFormalCharge(),
        atom.GetNumImplicitHs(),
        int(atom.IsInRing())
    ]
    return features

def get_bond_features(bond):
    """边级别特征（论文Table 2增强版）"""
    return [
        bond.GetBondTypeAsDouble(),
        int(bond.GetIsConjugated()),
        int(bond.IsInRing()),
        int(bond.GetStereo() != Chem.rdchem.BondStereo.STEREONONE),
        bond.GetBondDir().real
    ]

def create_graph_from_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: 
        print(f"Invalid SMILES: {smiles}")
        return None
    
    # 节点特征
    x = torch.tensor([get_atom_features(atom) for atom in mol.GetAtoms()], dtype=torch.float)
    
    # 边索引和特征
    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        # 双向边
        edge_index.extend([[i, j], [j, i]])  
        edge_attr.extend([get_bond_features(bond)] * 2)
    
    return Data(
        x=x,
        edge_index=torch.tensor(edge_index).t().contiguous(),
        edge_attr=torch.tensor(edge_attr, dtype=torch.float)
    )

In [9]:
# 2. 改进的wD-MPNN模型 ================================================
class WDMPNN(MessagePassing):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__(aggr='mean')
        
        # 节点编码器
        self.node_encoder = nn.Sequential(
            nn.Linear(in_channels, hidden_channels),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        # 边编码器
        self.edge_encoder = nn.Sequential(
            nn.Linear(5, hidden_channels),  # 对应get_bond_features的5维
            nn.ReLU()
        )
        
        # 消息传递层 - 修正：用于message函数中的特征变换
        self.message_nn = nn.Sequential(
            nn.Linear(hidden_channels * 2, hidden_channels),  # 拼接节点和边特征
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        # 节点更新层
        self.update_nn = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        # 输出头
        self.node_decoder = nn.Linear(hidden_channels, in_channels)  # 用于节点特征重建
        self.edge_decoder = nn.Linear(hidden_channels*2, 5)         # 用于边特征重建
        self.graph_decoder = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels//2),
            nn.ReLU(),
            nn.Linear(hidden_channels//2, out_channels)
        )
        
    def build_conv(self, channels):
        return nn.Sequential(
            nn.Linear(channels*2, channels),  # 拼接节点和边特征
            nn.ReLU(),
            nn.Dropout(0.3)
        )
    
    def forward(self, x, edge_index, edge_attr, batch=None):    
        # 编码
        x = self.node_encoder(x)
        edge_attr = self.edge_encoder(edge_attr)
        
        # 第一层消息传递
        x = self.propagate(edge_index, x=x, edge_attr=edge_attr)
        x = self.update_nn(x)

        # 第二层消息传递
        x = self.propagate(edge_index, x=x, edge_attr=edge_attr)
        x = self.update_nn(x)
        
        # 多任务输出
        node_out = self.node_decoder(x)
        # 边特征重建
        edge_out = self.edge_decoder(torch.cat([
            x[edge_index[0]], 
            x[edge_index[1]]
        ], dim=-1))
        
        # 图级别预测
        if batch is not None:
            graph_out = self.graph_decoder(global_mean_pool(x, batch))
            return node_out, edge_out, graph_out
        else:
            # 如果没有batch，假设是单个图
            graph_out = self.graph_decoder(x.mean(dim=0, keepdim=True))
            return node_out, edge_out, graph_out

    def message(self, x_j, edge_attr):
        # x_j: 邻居节点特征
        # edge_attr: 边特征
        return self.message_nn(torch.cat([x_j, edge_attr], dim=-1))

In [10]:
# 3. 自监督预训练 =====================================================
def mask_data(data, node_mask_ratio=0.15, edge_mask_ratio=0.15):
    """增强的掩码函数（论文Section 2.3.1）"""
    data = deepcopy(data)
    
    # 节点掩码
    num_nodes = data.x.size(0)
    node_mask = torch.rand(num_nodes) < node_mask_ratio
    data.masked_nodes = node_mask
    data.original_x = data.x.clone()
    data.x[node_mask] = 0
    
    # 边掩码
    num_edges = data.edge_index.size(1)
    edge_mask = torch.rand(num_edges) < edge_mask_ratio
    data.masked_edges = edge_mask
    data.original_edge_attr = data.edge_attr.clone()
    data.edge_attr[edge_mask] = 0
    
    return data

def pretrain(model, graph_data, epochs=100, patience=5):
    """自监督预训练"""
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = ReduceLROnPlateau(optimizer, 'min', patience=2, factor=0.5)
    best_loss = float('inf')
    no_improve = 0
    
    print("开始预训练...")
    
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        
        # 掩码并批处理
        masked_data = [mask_data(g) for g in graph_data]
        
        # 验证数据有效性
        valid_data = []
        for i, data in enumerate(masked_data):
            if data.x.size(0) > 0 and data.edge_index.size(1) > 0:
                valid_data.append(data)
            else:
                print(f"警告: 第{i}个图数据无效，跳过")
        
        if len(valid_data) == 0:
            print("错误: 没有有效的图数据")
            break
            
        try:
            batch_obj = Batch.from_data_list(valid_data)
            print(f"Epoch {epoch+1}: 批处理了 {len(valid_data)} 个图")
            
            # 前向传播
            node_pred, edge_pred, _ = model(
                batch_obj.x, 
                batch_obj.edge_index, 
                batch_obj.edge_attr, 
                batch_obj.batch
            )
            
            # 收集所有掩码信息
            all_masked_nodes = []
            all_masked_edges = []
            all_original_x = []
            all_original_edge_attr = []
            
            node_offset = 0
            edge_offset = 0
            
            for data in valid_data:
                # 节点掩码（考虑批处理后的索引偏移）
                masked_node_indices = torch.where(data.masked_nodes)[0] + node_offset
                all_masked_nodes.append(masked_node_indices)
                all_original_x.append(data.original_x[data.masked_nodes])
                
                # 边掩码（考虑批处理后的索引偏移）
                masked_edge_indices = torch.where(data.masked_edges)[0] + edge_offset
                all_masked_edges.append(masked_edge_indices)
                all_original_edge_attr.append(data.original_edge_attr[data.masked_edges])
                
                node_offset += data.x.size(0)
                edge_offset += data.edge_index.size(1)
            
            # 合并掩码信息
            if all_masked_nodes and any(len(mask) > 0 for mask in all_masked_nodes):
                masked_node_indices = torch.cat([mask for mask in all_masked_nodes if len(mask) > 0])
                original_node_features = torch.cat([feat for feat in all_original_x if len(feat) > 0])
                node_loss = F.mse_loss(node_pred[masked_node_indices], original_node_features)
            else:
                node_loss = torch.tensor(0.0, requires_grad=True)
            
            if all_masked_edges and any(len(mask) > 0 for mask in all_masked_edges):
                masked_edge_indices = torch.cat([mask for mask in all_masked_edges if len(mask) > 0])
                original_edge_features = torch.cat([feat for feat in all_original_edge_attr if len(feat) > 0])
                edge_loss = F.mse_loss(edge_pred[masked_edge_indices], original_edge_features)
            else:
                edge_loss = torch.tensor(0.0, requires_grad=True)
            
            # 总损失
            loss = node_loss + 0.5 * edge_loss
            
            # 反向传播
            loss.backward()
            optimizer.step()
            scheduler.step(loss)
            
            # 早停机制
            if loss < best_loss:
                best_loss = loss
                no_improve = 0
                torch.save(model.state_dict(), 'best_pretrain.pt')
            else:
                no_improve += 1
                if no_improve >= patience:
                    print(f"Early stopping at epoch {epoch+1}")
                    break
            
            print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f} | Node Loss: {node_loss.item():.4f} | Edge Loss: {edge_loss.item():.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")
            
        except Exception as e:
            print(f"Epoch {epoch+1} 出错: {e}")
            continue
    
    print("预训练完成!")


In [11]:
def finetune(
    model,
    graph_data,
    targets,
    epochs=200,
    lr=0.005,
    patience=10,
    prefix='model',
    scheduler_patience=3,
    scheduler_factor=0.5
):
    """监督微调函数：支持传入超参数 lr / patience / scheduler"""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = ReduceLROnPlateau(optimizer, 'min', patience=scheduler_patience, factor=scheduler_factor)
    best_loss = float('inf')
    no_improve = 0
    
    print("开始微调...")
    
    if len(graph_data) != len(targets):
        raise ValueError(f"图数据数量({len(graph_data)})与目标数量({len(targets)})不匹配")
    
    if not isinstance(targets, torch.Tensor):
        targets = torch.tensor(targets, dtype=torch.float32)
    if len(targets.shape) == 1:
        targets = targets.unsqueeze(-1)

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        try:
            valid_data = []
            valid_targets = []
            for i, (data, target) in enumerate(zip(graph_data, targets)):
                if data.x.size(0) == 0 or data.edge_index.size(1) == 0:
                    print(f"警告: 第{i}个图数据无效，跳过")
                    continue
                if torch.isnan(data.x).any() or torch.isinf(data.x).any():
                    print(f"警告: 第{i}个图的节点特征包含NaN/Inf，跳过")
                    continue
                if torch.isnan(target).any() or torch.isinf(target).any():
                    print(f"警告: 第{i}个目标包含NaN/Inf，跳过")
                    continue
                valid_data.append(data)
                valid_targets.append(target)

            if len(valid_data) == 0:
                print("错误: 没有有效的图数据")
                break

            valid_targets = torch.stack(valid_targets) if len(valid_targets) > 1 else valid_targets[0].unsqueeze(0)

            batch_obj = Batch.from_data_list(valid_data)
            _, _, graph_pred = model(
                batch_obj.x,
                batch_obj.edge_index,
                batch_obj.edge_attr,
                batch_obj.batch
            )

            loss = F.mse_loss(graph_pred, valid_targets)
            loss.backward()
            optimizer.step()
            scheduler.step(loss)

            if loss < best_loss:
                best_loss = loss
                no_improve = 0
                torch.save(model.state_dict(), f'{prefix}_best_finetune.pt')
            else:
                no_improve += 1
                if no_improve >= patience:
                    print(f"Early stopping at epoch {epoch+1}")
                    break

            print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

        except Exception as e:
            print(f"Epoch {epoch+1} 出错: {e}")
            import traceback
            traceback.print_exc()
            continue

    print("微调完成!")
    print(f"最佳损失: {best_loss:.4f}")

In [12]:
# 设备选择
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'device:{device}')

device:cpu


In [13]:
# 数据准备
graph_data = [create_graph_from_smiles(s).to(device) for s in dev_train['SMILES']]
graph_data = [g.to(device) for g in graph_data if g is not None]
#targets = torch.tensor(dev_train[['Tg','FFV','Tc','Density','Rg']].values, dtype=torch.float32)

In [14]:
# 模型初始化
model = WDMPNN(
    in_channels=6,  # 对应get_atom_features的输出维度
    hidden_channels=256,
    out_channels=1  # 1个预测目标
).to(device)

In [15]:
# 自监督预训练（论文Phase 1）
print("=== Self-supervised Pretraining ===")
pretrain(model, graph_data, epochs=100)

=== Self-supervised Pretraining ===
开始预训练...
Epoch 1: 批处理了 4783 个图
Epoch 1/100 | Loss: 8.3795 | Node Loss: 8.0669 | Edge Loss: 0.6251 | LR: 0.001000
Epoch 2: 批处理了 4783 个图
Epoch 2/100 | Loss: 8.0258 | Node Loss: 7.7307 | Edge Loss: 0.5902 | LR: 0.001000
Epoch 3: 批处理了 4783 个图
Epoch 3/100 | Loss: 7.6593 | Node Loss: 7.3869 | Edge Loss: 0.5448 | LR: 0.001000
Epoch 4: 批处理了 4783 个图
Epoch 4/100 | Loss: 6.9231 | Node Loss: 6.6895 | Edge Loss: 0.4672 | LR: 0.001000
Epoch 5: 批处理了 4783 个图
Epoch 5/100 | Loss: 5.9878 | Node Loss: 5.8133 | Edge Loss: 0.3489 | LR: 0.001000
Epoch 6: 批处理了 4783 个图
Epoch 6/100 | Loss: 4.6088 | Node Loss: 4.4874 | Edge Loss: 0.2429 | LR: 0.001000
Epoch 7: 批处理了 4783 个图
Epoch 7/100 | Loss: 3.1193 | Node Loss: 2.9103 | Edge Loss: 0.4179 | LR: 0.001000
Epoch 8: 批处理了 4783 个图
Epoch 8/100 | Loss: 2.3749 | Node Loss: 1.6877 | Edge Loss: 1.3745 | LR: 0.001000
Epoch 9: 批处理了 4783 个图
Epoch 9/100 | Loss: 2.9363 | Node Loss: 1.9474 | Edge Loss: 1.9778 | LR: 0.001000
Epoch 10: 批处理了 4783

In [16]:
# 读取训练数据
train_df = pd.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/train.csv')

# 提取图数据
graph_data = [create_graph_from_smiles(s).to(device) for s in train_df['SMILES']]
graph_data = [g.to(device) for g in graph_data if g is not None]

# 目标列
target_columns = ['Tg', 'FFV', 'Tc', 'Density', 'Rg']

# 对每个目标列训练一个模型
for target in target_columns:
    # 提取目标列数据并处理NaN
    targets = torch.tensor(train_df[target].values, dtype=torch.float32).view(-1, 1)

    # 对当前目标列执行 dropna
    train_df_target_clean = train_df.dropna(subset=[target])  # 只针对当前目标列进行 NaN 处理
    train_df_target_clean = train_df_target_clean[~train_df_target_clean[target].isin([float('inf'), -float('inf')])]  # 删除 Inf 数据

    # 重新生成图数据
    graph_data_clean = [create_graph_from_smiles(s).to(device) for s in train_df_target_clean['SMILES']]
    graph_data_clean = [g.to(device) for g in graph_data_clean if g is not None]

    # 重新提取目标数据
    targets_clean = torch.tensor(train_df_target_clean[target].values, dtype=torch.float32).view(-1, 1).to(device)

    # 确保图数据和目标数据长度一致
    valid_data = []
    valid_targets = []
    for i, (data, target_value) in enumerate(zip(graph_data_clean, targets_clean)):
        if data.x.size(0) > 0 and data.edge_index.size(1) > 0:  # 确保图数据有效
            valid_data.append(data)
            valid_targets.append(target_value)

    if len(valid_data) == 0:
        print(f"没有有效的图数据用于训练 {target}")
        continue

    # 加载预训练的权重
    print(f"加载预训练的权重用于 {target}...")
    model.load_state_dict(torch.load('best_pretrain.pt'))  # 加载预训练模型

    # 开始微调
    print(f"Training model for {target}...")
    
    # 调用原始的finetune函数进行训练
    finetune(model, valid_data, torch.stack(valid_targets), epochs=2000, prefix=target)

    print(f"Finished training model for {target}")

加载预训练的权重用于 Tg...
Training model for Tg...
开始微调...
Epoch 1/2000 | Loss: 21727.9746 | LR: 0.005000
Epoch 2/2000 | Loss: 21432.3906 | LR: 0.005000
Epoch 3/2000 | Loss: 20342.5156 | LR: 0.005000
Epoch 4/2000 | Loss: 16048.5381 | LR: 0.005000
Epoch 5/2000 | Loss: 13277.5879 | LR: 0.005000
Epoch 6/2000 | Loss: 11610.4238 | LR: 0.005000
Epoch 7/2000 | Loss: 11809.3691 | LR: 0.005000
Epoch 8/2000 | Loss: 11424.2324 | LR: 0.005000
Epoch 9/2000 | Loss: 10818.6348 | LR: 0.005000
Epoch 10/2000 | Loss: 10939.1445 | LR: 0.005000
Epoch 11/2000 | Loss: 10149.4150 | LR: 0.005000
Epoch 12/2000 | Loss: 10043.8066 | LR: 0.005000
Epoch 13/2000 | Loss: 9763.1299 | LR: 0.005000
Epoch 14/2000 | Loss: 9142.0732 | LR: 0.005000
Epoch 15/2000 | Loss: 9089.4082 | LR: 0.005000
Epoch 16/2000 | Loss: 8546.1572 | LR: 0.005000
Epoch 17/2000 | Loss: 8215.5186 | LR: 0.005000
Epoch 18/2000 | Loss: 7961.7046 | LR: 0.005000
Epoch 19/2000 | Loss: 7544.9600 | LR: 0.005000
Epoch 20/2000 | Loss: 7558.9902 | LR: 0.005000
Epoch 2

In [ ]:
# ====== 微调 Tg 模型 ======
target = 'Tg'
target_config = {'epochs': 1500, 'lr': 0.001, 'patience': 20}

# 清洗训练数据
train_df_clean = train_df.dropna(subset=[target])
train_df_clean = train_df_clean[~train_df_clean[target].isin([float('inf'), -float('inf')])]
graph_data_clean = [create_graph_from_smiles(s).to(device) for s in train_df_clean['SMILES']]
graph_data_clean = [g for g in graph_data_clean if g is not None]
targets_clean = torch.tensor(train_df_clean[target].values, dtype=torch.float32).view(-1, 1).to(device)

# 筛选有效数据
valid_data, valid_targets = [], []
for g, t in zip(graph_data_clean, targets_clean):
    if g.x.size(0) > 0 and g.edge_index.size(1) > 0:
        valid_data.append(g)
        valid_targets.append(t)
if not valid_data:
    raise RuntimeError("No valid training data for Tg")

# 重置模型并加载预训练
model.load_state_dict(torch.load('best_pretrain.pt'))

# 训练
finetune(
    model=model,
    graph_data=valid_data,
    targets=torch.stack(valid_targets),
    epochs=target_config['epochs'],
    lr=target_config['lr'],
    patience=target_config['patience'],
    prefix=target
)

# ====== 评估 Tg 模型 ======
# 获取测试图数据
X_test = [create_graph_from_smiles(s).to(device) for s in dev_test['SMILES']]
X_test = [g.to(device) for g in X_test if g is not None]

# 加载模型
model.load_state_dict(torch.load(f'{target}_best_finetune.pt'))
model.eval()

# 提取目标数据
target_data = dev_test[target].dropna()
targets = torch.tensor(target_data.values, dtype=torch.float32).view(-1, 1).to(device)
valid_indices = target_data.index
valid_data = [X_test[i] for i in valid_indices if i < len(X_test)]
targets = targets[:len(valid_data)]

# 预测
with torch.no_grad():
    batch_obj = Batch.from_data_list(valid_data)
    _, _, graph_pred = model(
        batch_obj.x,
        batch_obj.edge_index,
        batch_obj.edge_attr,
        batch_obj.batch
    )

# 计算 MSE
mask = ~torch.isnan(targets)
mse = mean_squared_error(
    targets[mask].view(-1).cpu().numpy(),
    graph_pred[mask.view(-1)].view(-1).cpu().numpy()
)
print(f"\n>>> {target} MSE: {mse:.4f}")

In [17]:
# 4. 获取测试数据的图数据
X_test = [create_graph_from_smiles(s).to(device) for s in dev_test['SMILES']]
X_test = [g.to(device) for g in X_test if g is not None]

In [18]:
# 4. 针对每个目标列加载权重，进行预测并计算 MSE
mse_per_task = {}
for target in target_columns:
    print(f"Evaluating model for {target}...")

    # 读取相应的权重文件
    model.load_state_dict(torch.load(f'{target}_best_finetune.pt'))

    # 设置模型为评估模式
    model.eval()

    # 提取目标数据，并对当前目标列执行 dropna
    target_data = dev_test[target].dropna()
    targets = torch.tensor(target_data.values, dtype=torch.float32).view(-1, 1).to(device)
    
    # 获取对应的图数据 - 确保与目标数据对齐
    valid_indices = target_data.index
    valid_data = [X_test[i].to(device) for i in valid_indices if i < len(X_test) and X_test[i] is not None]
    
    # 检查数据对齐
    if len(valid_data) != len(targets):
        print(f"Warning: {len(targets)} targets but {len(valid_data)} graphs for {target}")
        # 只保留有对应图的目标值
        targets = targets[:len(valid_data)]
    
    # 检查有效数据
    if len(valid_data) == 0:
        print(f"No valid data for target {target}")
        mse_per_task[target] = float('nan')
        continue
    
    # 使用 Batch.from_data_list 来处理不同大小的图数据
    batch_obj = Batch.from_data_list(valid_data)

    # 进行预测 - 我们只需要图级别的预测（第三个返回值）
    with torch.no_grad():
        _, _, graph_predictions = model(
            batch_obj.x, 
            batch_obj.edge_index, 
            batch_obj.edge_attr, 
            batch_obj.batch
        )

    # 确保预测和目标数量匹配
    if len(graph_predictions) != len(targets):
        min_len = min(len(graph_predictions), len(targets))
        graph_predictions = graph_predictions[:min_len]
        targets = targets[:min_len]
    
    # 计算 MSE
    mask = ~torch.isnan(targets)
    if mask.sum() > 0:
        # 确保形状匹配
        valid_targets = targets[mask].view(-1)
        valid_predictions = graph_predictions[mask.view(-1)].view(-1)

        mse = mean_squared_error(
            valid_targets.cpu().numpy(), 
            valid_predictions.cpu().numpy()
        )
        mse_per_task[target] = mse
    else:
        mse_per_task[target] = float('nan')

    print(f"  Processed {len(graph_predictions)} predictions for {target}")

# 输出每个任务的 MSE
print("\nMSE per task:")
for name, mse in mse_per_task.items():
    print(f"  {name}: {mse:.4f}")

# 计算平均 MSE（仅计算有有效值的任务）
valid_mses = [mse for mse in mse_per_task.values() if not np.isnan(mse)]
if valid_mses:
    overall_mse = np.mean(valid_mses)
    print(f"\nAverage MSE across tasks: {overall_mse:.4f}")
else:
    print("\nNo valid MSE values to compute overall average")

Evaluating model for Tg...
  Processed 17 predictions for Tg
Evaluating model for FFV...
  Processed 297 predictions for FFV
Evaluating model for Tc...
  Processed 33 predictions for Tc
Evaluating model for Density...
  Processed 20 predictions for Density
Evaluating model for Rg...
  Processed 21 predictions for Rg

MSE per task:
  Tg: 27960.7070
  FFV: 0.0018
  Tc: 0.0075
  Density: 0.0607
  Rg: 112.6207

Average MSE across tasks: 5614.6797


In [19]:
# 1. Load sample submission template
sample_path = '/kaggle/input/neurips-open-polymer-prediction-2025/test.csv'  # Update with actual path
submission_df = pd.read_csv(sample_path)
print(f"Loaded sample submission with {len(submission_df)} SMILES strings")

# 2. Precompute graphs for all submission SMILES
smiles_list = submission_df['SMILES'].tolist()
graph_list = []
valid_indices = []  # Track which indices have valid graphs

print("Creating molecular graphs for submission SMILES...")
for idx, smiles in enumerate(smiles_list):
    graph = create_graph_from_smiles(smiles)
    if graph is not None:
        graph = graph.to(device)
        graph_list.append(graph)
        valid_indices.append(idx)

print(f"Successfully created {len(graph_list)}/{len(smiles_list)} molecular graphs")

# 3. Batch valid graphs if any exist
if graph_list:
    batch_obj = Batch.from_data_list(graph_list).to(device)
    print(f"Created batch with {batch_obj.num_graphs} graphs")
else:
    print("Warning: No valid graphs created - submission will be all NaN")
    batch_obj = None

# 4. Make predictions for each target property
for target in target_columns:
    print(f"\nPredicting {target}...")
    
    try:
        # Load fine-tuned weights for this property
        model_path = f'{target}_best_finetune.pt'
        model.load_state_dict(torch.load(model_path))
        model.eval()
        print(f"  Loaded weights from {model_path}")
        
        # Make predictions if we have valid graphs
        if batch_obj:
            with torch.no_grad():
                _, _, graph_preds = model(
                    batch_obj.x,
                    batch_obj.edge_index,
                    batch_obj.edge_attr,
                    batch_obj.batch
                )
            
            # Convert predictions to numpy array
            predictions = graph_preds.cpu().numpy().flatten()
            
            # Fill predictions into valid positions
            for i, idx in enumerate(valid_indices):
                submission_df.at[idx, target] = predictions[i]
            print(f"  Added predictions for {len(predictions)} molecules")
        else:
            print("  No valid graphs - skipping prediction")
            
    except Exception as e:
        print(f"  Error predicting {target}: {str(e)}")

# 5. Drop the SMILES column from the final submission
submission_df = submission_df.drop(columns=['SMILES'])

# 6. Save final submission
submission_df.to_csv('submission.csv',index=False)
print(f"\nSubmission saved to submission.csv")
print("Submission preview:")
print(submission_df.head())

# 7. Show statistics
print("\nSubmission summary:")
for target in target_columns:
    num_predicted = submission_df[target].notna().sum()
    print(f"  {target}: {num_predicted}/{len(submission_df)} predicted")

Loaded sample submission with 3 SMILES strings
Creating molecular graphs for submission SMILES...
Successfully created 3/3 molecular graphs
Created batch with 3 graphs

Predicting Tg...
  Loaded weights from Tg_best_finetune.pt
  Added predictions for 3 molecules

Predicting FFV...
  Loaded weights from FFV_best_finetune.pt
  Added predictions for 3 molecules

Predicting Tc...
  Loaded weights from Tc_best_finetune.pt
  Added predictions for 3 molecules

Predicting Density...
  Loaded weights from Density_best_finetune.pt
  Added predictions for 3 molecules

Predicting Rg...
  Loaded weights from Rg_best_finetune.pt
  Added predictions for 3 molecules

Submission saved to submission.csv
Submission preview:
           id          Tg       FFV        Tc   Density        Rg
0  1109053969  150.004196  0.341185  0.238356  1.137323  5.730541
1  1422188626  191.808075  0.342410  0.235713  1.171559  5.824758
2  2032016830  163.779129  0.340256  0.236290  1.150849  5.745608

Submission summary: